# Data Cleaning and Outlier Analysis
This notebook analyzes 'strange cases' in the Inibsa dataset, specifically:
1. Clients with billing significantly higher than their estimated potential.
2. Clients with very low transaction frequency.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Settings
sns.set(style='whitegrid')
%matplotlib inline

## 1. Loading Data

In [ ]:
df_commodities = pd.read_csv('data/master_commodities.csv', low_memory=False)
df_technicals = pd.read_csv('data/master_technicals.csv', low_memory=False)

df_commodities['Fecha'] = pd.to_datetime(df_commodities['Fecha'])
df_technicals['Fecha'] = pd.to_datetime(df_technicals['Fecha'])

print(f'Commodities: {len(df_commodities)} rows')
print(f'Technicals: {len(df_technicals)} rows')

## 2. Billing vs Potential Analysis
We compare the annual billing per client and family against the estimated annual potential.

In [ ]:
def analyze_billing_vs_potential(df, title):
    # Aggregate by Client, Year and Family
    df['anyo'] = df['Fecha'].dt.year
    annual = df.groupby(['Id_Cliente', 'anyo', 'Familia_Potencial']).agg({
        'Valores_H': 'sum',
        'Potencial_EUR': 'first'
    }).reset_index()
    
    # Ratio
    annual['Ratio'] = annual['Valores_H'] / annual['Potencial_EUR']
    annual.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Filter strange cases (e.g., Ratio > 3)
    strange = annual[annual['Ratio'] > 3]
    
    print(f'--- {title} ---')
    print(f'Total Client-Year-Family records: {len(annual)}')
    print(f'Records with billing > 3x Potential: {len(strange)}')
    
    # Visualization
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=annual, x='Potencial_EUR', y='Valores_H', alpha=0.5)
    plt.plot([0, annual['Potencial_EUR'].max()], [0, annual['Potencial_EUR'].max()], 'r--', label='Billing = Potential')
    plt.plot([0, annual['Potencial_EUR'].max()], [0, 3*annual['Potencial_EUR'].max()], 'g--', label='Billing = 3x Potential')
    plt.title(f'Billing vs Potential ({title})')
    plt.legend()
    plt.show()
    
    return strange

strange_comm = analyze_billing_vs_potential(df_commodities, 'Commodities')
strange_tech = analyze_billing_vs_potential(df_technicals, 'Technicals')

## 3. Transaction Frequency Analysis
Identifying clients with very few invoices over the 5-year period.

In [ ]:
def analyze_frequency(df, title):
    freq = df.groupby('Id_Cliente')['Num.Fact'].nunique().reset_index()
    freq.columns = ['Id_Cliente', 'Invoice_Count']
    
    # Filter low frequency (e.g., < 3 invoices in 5 years)
    low_freq = freq[freq['Invoice_Count'] < 3]
    
    print(f'--- {title} ---')
    print(f'Total unique clients: {len(freq)}')
    print(f'Clients with < 3 invoices: {len(low_freq)}')
    
    # Visualization
    plt.figure(figsize=(10, 6))
    sns.histplot(freq['Invoice_Count'], bins=50, kde=True)
    plt.axvline(3, color='r', linestyle='--', label='Threshold (< 3)')
    plt.title(f'Distribution of Invoices per Client ({title})')
    plt.legend()
    plt.show()
    
    return low_freq

low_freq_comm = analyze_frequency(df_commodities, 'Commodities')
low_freq_tech = analyze_frequency(df_technicals, 'Technicals')

## 4. Cleaning Strategy
We will remove:
1. Clients with annual billing > 5x Potential (extreme outliers).
2. Clients with < 3 total invoices (insufficient history).

In [ ]:
def clean_dataset(df, strange, low_freq):
    # Extreme outliers (Ratio > 5)
    extreme_clients = strange[strange['Ratio'] > 5]['Id_Cliente'].unique()
    # Low frequency clients
    low_freq_clients = low_freq['Id_Cliente'].unique()
    
    exclude_clients = set(extreme_clients) | set(low_freq_clients)
    
    df_clean = df[~df['Id_Cliente'].isin(exclude_clients)].copy()
    
    print(f'Original rows: {len(df)}')
    print(f'Cleaned rows: {len(df_clean)}')
    print(f'Removed clients: {len(exclude_clients)}')
    
    return df_clean

df_commodities_clean = clean_dataset(df_commodities, strange_comm, low_freq_comm)
df_technicals_clean = clean_dataset(df_technicals, strange_tech, low_freq_tech)

## 5. Saving Cleaned Data

In [ ]:
df_commodities_clean.to_csv('data/master_commodities_clean.csv', index=False)
df_technicals_clean.to_csv('data/master_technicals_clean.csv', index=False)
print('Datasets saved successfully.')